In [18]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader

with open("the-verdict.txt", 'r') as file:
    raw_text = file.read()

In [19]:
class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        self.token_ids = token_ids

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def CreateDataloader(txt, batch_size=4, max_length=256,
                     stride=128, shuffle=True, drop_last=True,
                     num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last,
    )

    return dataloader

In [3]:
dataloader = CreateDataloader(raw_text, batch_size=8, max_length=8, stride=8, shuffle=True)
data_iter = iter(dataloader)


In [4]:
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)


Inputs:
 tensor([[ 9546,   546,   262, 15393,   852,  4808,  3810,    62],
        [  340,   262, 11644, 43778,  3465,   326, 26773,   606],
        [ 6164,    25,   366, 16773,   290,   766,   262,  1334],
        [  271, 10899,   550,   366,  7109, 14655,   683,   866],
        [  292,   314, 10226,   262,  1182,   287,   314,  2936],
        [  284,   502,   262,  4112,   957,  1483,   286,  3619],
        [  645, 15223,   670,   286,  8166,   438, 14363,  1986],
        [  319,   465,   636,    11,   257, 29844,   286, 12749]])

Targets:
 tensor([[  546,   262, 15393,   852,  4808,  3810,    62,   438],
        [  262, 11644, 43778,  3465,   326, 26773,   606,   286],
        [   25,   366, 16773,   290,   766,   262,  1334,   286],
        [10899,   550,   366,  7109, 14655,   683,   866,   526],
        [  314, 10226,   262,  1182,   287,   314,  2936,   355],
        [  502,   262,  4112,   957,  1483,   286,  3619,   338],
        [15223,   670,   286,  8166,   438, 14363,  198

In [8]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_keys = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_values = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None
        self.register_buffer(
            "mask", torch.triu( 
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )
        

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        values = self.W_values(x)
        keys = self.W_keys(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        self.dropout(attn_weights)
        self.attn_weights = attn_weights

        context_vec = attn_weights @ values
        return context_vec
            

In [9]:
torch.manual_seed(123)
x = torch.randn(8, 8, 256)
ca = CausalAttention(256, 256, 8, 0)

out = ca(x)
out.shape

torch.Size([8, 8, 256])

In [10]:
print(ca.attn_weights.shape)

torch.Size([8, 8, 8])


In [11]:
w = ca.attn_weights[0]
print(w.round(decimals=2))
print(w.sum(dim=-1))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4800, 0.5200, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4200, 0.2100, 0.3700, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4600, 0.1600, 0.2000, 0.1800, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2600, 0.2100, 0.2000, 0.1100, 0.2200, 0.0000, 0.0000, 0.0000],
        [0.2400, 0.1500, 0.1200, 0.1700, 0.2100, 0.1000, 0.0000, 0.0000],
        [0.0600, 0.2400, 0.1300, 0.1100, 0.1100, 0.2100, 0.1400, 0.0000],
        [0.0700, 0.1500, 0.1400, 0.1300, 0.1500, 0.0900, 0.1000, 0.1700]],
       grad_fn=<RoundBackward1>)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


In [12]:
class MiniAttentionModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_length, dropout):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(context_length, emb_dim)
        self.attn = CausalAttention(emb_dim, emb_dim, context_length, dropout)
        self.out_head = nn.Linear(emb_dim, vocab_size)


    def forward(self, ids):
        b, num_tokens = ids.shape
        tok = self.tok_emb(ids)
        pos = self.pos_emb(torch.arange(num_tokens, device=ids.device))
        x = tok + pos
        x = self.attn(x)
        logits = self.out_head(x)
        return logits

In [13]:
torch.manual_seed(123)
model = MiniAttentionModel(vocab_size=50257, emb_dim=256, context_length=8, dropout=0.1)
logits = model(inputs)
print(logits.shape)

torch.Size([8, 8, 50257])


In [14]:
logits

tensor([[[ 3.2404e-01, -6.2385e-01,  1.9777e-01,  ..., -4.2431e-01,
          -5.2039e-01,  9.9782e-02],
         [ 2.0059e-02, -3.7419e-01,  1.0223e-01,  ..., -3.0908e-01,
          -3.2868e-01, -1.1775e-01],
         [ 1.1701e-01, -1.6685e-01,  5.8513e-02,  ..., -1.4376e-01,
          -2.2458e-01, -1.2306e-01],
         ...,
         [ 1.1142e-02,  4.1629e-01,  6.5079e-02,  ...,  7.0774e-02,
           1.8355e-01, -4.5640e-02],
         [ 3.3823e-02,  2.5094e-01,  4.0738e-02,  ..., -1.6186e-02,
          -3.2981e-02,  2.7877e-03],
         [-4.9441e-02,  2.4911e-01,  9.3595e-02,  ..., -1.3261e-01,
          -1.0278e-02, -1.9215e-02]],

        [[ 3.5191e-02,  8.3702e-02,  5.1862e-01,  ..., -1.1655e+00,
          -8.8336e-02,  1.6946e-01],
         [ 8.0672e-02,  2.5007e-01,  1.0455e-01,  ..., -7.7814e-01,
          -5.0186e-03,  4.4456e-02],
         [ 9.3906e-02,  3.0787e-01, -1.4342e-01,  ..., -1.5817e-01,
           1.4978e-01, -1.5662e-01],
         ...,
         [ 1.1423e-01,  2

In [16]:
with open("the-verdict.txt", "r") as f:
    raw_text = f.read()

print(type(raw_text))
print(len(raw_text))
print(raw_text[:100])

<class 'str'>
20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [17]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(raw_text)

print(type(token_ids))
print(len(token_ids))
print(token_ids[:10])
print([tokenizer.decode([t]) for t in token_ids[:10]])


<class 'list'>
5145
[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
['I', ' H', 'AD', ' always', ' thought', ' Jack', ' G', 'is', 'burn', ' rather']
